In [1]:
import pandas as pd

# Clientes
clientes = pd.DataFrame({
    "id_cliente": [1, 2, 3],
    "nombre_cliente": ["Ana", "Luis", "Carlos"],
    "ciudad": ["Bogotá", "Medellín", "Cali"]
})

# Ventas
ventas = pd.DataFrame({
    "id_venta": [101, 102, 103, 104],  # nota: 104 no tiene cliente
    "fecha": ["2025-10-01", "2025-10-02", "2025-10-03", "2025-10-04"],
    "id_cliente": [1, 2, 99, 3]  # id_cliente=99 no existe en clientes
})

# Productos
productos = pd.DataFrame({
    "id_producto": [90, 82, 39],
    "nombre_producto": ["Zapatos", "Camisa", "Pantalón"],
    "precio": [2902, 2394, 1500]
})

# Detalle de venta
detalle_venta = pd.DataFrame({
    "id_detalle_venta": [1, 2, 3, 4],
    "id_venta": [101, 102, 103, 105],  # id_venta=105 no existe en ventas
    "id_producto": [90, 82, 999, 39],  # id_producto=999 no existe en productos
    "cantidad": [1, 5, 2, 1],
    "importe": [2902, 11970, 3000, 1500]
})



In [10]:
# Unimos detalle con ventas con outer
check_ventas = pd.merge(detalle_venta, ventas, on="id_venta", how="outer")
print("Inconsistencias entre detalle_venta y ventas:")
display(check_ventas)

# Unimos detalle con productos con outer
check_productos = pd.merge(detalle_venta, productos, on="id_producto", how="outer")
print("\nInconsistencias entre detalle_venta y productos:")
display(check_productos)

# Unimos ventas con clientes con outer
check_clientes = pd.merge(ventas, clientes, on="id_cliente", how="outer")
print("\nInconsistencias entre ventas y clientes:")
display(check_clientes)


Inconsistencias entre detalle_venta y ventas:


,id_detalle_venta,id_venta,id_producto,cantidad,importe,fecha,id_cliente
0,1.0,101,90.0,1.0,2902.0,2025-10-01,1.0
1,2.0,102,82.0,5.0,11970.0,2025-10-02,2.0
2,3.0,103,999.0,2.0,3000.0,2025-10-03,99.0
3,NaN,104,NaN,NaN,NaN,2025-10-04,3.0
4,4.0,105,39.0,1.0,1500.0,NaN,NaN



Inconsistencias entre detalle_venta y productos:


,id_detalle_venta,id_venta,id_producto,cantidad,importe,nombre_producto,precio
0,4,105,39,1,1500,Pantalón,1500.0
1,2,102,82,5,11970,Camisa,2394.0
2,1,101,90,1,2902,Zapatos,2902.0
3,3,103,999,2,3000,NaN,NaN



Inconsistencias entre ventas y clientes:


,id_venta,fecha,id_cliente,nombre_cliente,ciudad
0,101,2025-10-01,1,Ana,Bogotá
1,102,2025-10-02,2,Luis,Medellín
2,104,2025-10-04,3,Carlos,Cali
3,103,2025-10-03,99,NaN,NaN


In [9]:
# Detalle + Ventas
df_merge1 = pd.merge(detalle_venta, ventas, on="id_venta", how="left")

# + Clientes
df_merge2 = pd.merge(df_merge1, clientes, on="id_cliente", how="left")

# + Productos
df_final = pd.merge(df_merge2, productos, on="id_producto", how="left")

print("\nDataset final para análisis:")
display(df_final)



Dataset final para análisis:


,id_detalle_venta,id_venta,id_producto,cantidad,importe,fecha,id_cliente,nombre_cliente,ciudad,nombre_producto,precio
0,1,101,90,1,2902,2025-10-01,1.0,Ana,Bogotá,Zapatos,2902.0
1,2,102,82,5,11970,2025-10-02,2.0,Luis,Medellín,Camisa,2394.0
2,3,103,999,2,3000,2025-10-03,99.0,NaN,NaN,NaN,NaN
3,4,105,39,1,1500,NaN,NaN,NaN,NaN,Pantalón,1500.0


In [4]:
df_final.isnull().sum()



id_detalle_venta    0
id_venta            0
id_producto         0
cantidad            0
importe             0
fecha               1
id_cliente          1
nombre_cliente      2
ciudad              2
nombre_producto     1
precio              1
dtype: int64

In [5]:
# 2. Filtrar solo las filas con NaN
df_final[df_final.isnull().any(axis=1)]


,id_detalle_venta,id_venta,id_producto,cantidad,importe,fecha,id_cliente,nombre_cliente,ciudad,nombre_producto,precio
2,3,103,999,2,3000,2025-10-03,99.0,NaN,NaN,NaN,NaN
3,4,105,39,1,1500,NaN,NaN,NaN,NaN,Pantalón,1500.0


In [6]:
# 🔹 3. Ver NaN en una columna específica
df_final[df_final["id_cliente"].isnull()]

,id_detalle_venta,id_venta,id_producto,cantidad,importe,fecha,id_cliente,nombre_cliente,ciudad,nombre_producto,precio
3,4,105,39,1,1500,NaN,NaN,NaN,NaN,Pantalón,1500.0


In [7]:
#🔹 4. Contar cuántos NaN hay en una columna
df = df_final["id_cliente"].isnull().sum()
print(df)

1


In [8]:
#exportar los registros problemáticos
errores = df_final[df_final.isnull().any(axis=1)]
errores.to_csv("inconsistencias.csv", index=False)
